In [40]:
!pip -q install transformers pandas

GPU and Drive

In [41]:
import os, json, numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "GPU missing: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))
from google.colab import drive; drive.mount("/content/drive")
D = "/content/drive/MyDrive/spoilfree/"
print(os.listdir(D))

Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['val.csv', 'test._heldout.csv', 'val_scored.csv', 'train_scored.csv', 'test_scored.csv', 'clean_train.csv', 'clean_val.csv', 'clean_test.csv', 'IMDB_reviews.json', 'IMDB_movie_details.json']


Settings

In [42]:
teacher="bhavyagiri/roberta-base-finetuned-imdb-spoilers"
student="roberta-base"
maxlen=160
keep_pos=0.30 #keep top 30% according to teacher
keep_neg=0.90 #keep 70% of the least spoiler content
epochs=3
bs=32
lr=2e-5
seed=1

try:
    test_file=D+"test_heldout.csv"
    if not os.path.exists(test_file):
      test_file=D+'test._heldout.csv'
    if not os.path.exists(test_file):
        raise FileNotFoundError("test._heldout.csv not found")
    print("Using:", test_file)
except FileNotFoundError as e:
    print(f"File not found: {e}")

torch.manual_seed(seed)
np.random.seed(seed)

Using: /content/drive/MyDrive/spoilfree/test._heldout.csv


**Helper Functions**

In [43]:
from sys import thread_info
from sklearn.metrics import roc_auc_score, average_precision_score

def clean_split(d,keep_posnew=None,keep_negnew=None,seed=1):
  keep_posnew=keep_pos if keep_posnew is None else keep_posnew
  keep_negnew=keep_neg if keep_negnew is None else keep_negnew
  pos,neg=d[d.label==1],d[d.label==0]
  pos=pos[pos.t1>=pos.t1.quantile(1-keep_posnew)]
  neg=neg[neg.t1>=neg.t1.quantile(keep_negnew)]
  k=min(len(pos),len(neg))
  out=pd.concat(pos.sample(k,random_state=seed),neg.sample(k,random_state=seed))
  return out.sample(frac=1,random_state=seed).reset_index(drop=True)[["text","label","t1"]]


def sweep(y,p,name,ths=(0.3,0.5,0.7,0.8,0.9,0.95)):
  y=np.asarray(y)
  p=np.asarray(p)
  print(f"{name} AUC: {roc_auc_score(y,p):.3f} AP {average_precision_score(y,p):.3f}")
  print(f"thr precision recall falseblur missed")
  for ts in ths:
    pr=p>=ts
    tp=int((pr & (y==1)).sum())
    fp=int((pr & (y==0)).sum())       #confusion matrix metric
    fn=int(((~pr) & (y==1)).sum())
    tn=int(((~pr) & (y==0)).sum())
    print(f"{ts:<6} {tp / max(tp + fp, 1):<10.3f} {tp / max(tp + fn, 1):<7.3f} {fp:<10} {fn}")


def pick_threshold(y,p,min_precision=0.95):
  #lowest precision that is >=min_precision if no threshold reaches then None
  y=np.asarray(y)
  p=np.asarray(p)
  for ts in np.arange(0.05,1.0,0.01):
    pr=p>=ts
    if pr.sum() and (pr&(y==1)).sum()/pr.sum()>=min_precision:
      return round(float(ts),2)
  return None




**Score from the teacher**

In [44]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
ttok=AutoTokenizer.from_pretrained(teacher)    #loads tokenizer to convert text into model readable tokens
teacher=AutoModelForSequenceClassification.from_pretrained(teacher).half().cuda().eval()  #loads pretrained teacher model
                                                                                            #then uses half precision to reduce memory usage,
                                                                                            #moves model to gpu and then enables evaluation mode
                                                                                            #disabling training spiecific behaviour like a dropout

@torch.no_grad()  #disables gradient calculations: saves memory and computation
def score_with(model,tokenizer,texts,bs=128):
  #processes upto 128 texts per batch


  lengths=[len(t) for t in texts]            #calculates the length of each text
  order=np.argsort(lengths)                    #sorts indices by text length in ascending order
  out=np.zeros(len(texts),dtype=np.float32)             #creates an array to store spoiler scores

  for i in range(0,len(texts),bs):                  #iterates through texts in batches of size bs
    j=order[i:i+bs]                                #gets sorted indices for the current batch
    batch_texts=[texts[k] for k in j]                     #retrieves texts using the sorted indices

    x=tokenizer(                             #converts batch texts into model-readable tokens
        batch_texts,                         #passes the current batch of texts
        truncation=True,                         #truncates texts exceeding the token limit
        max_length=maxlen,                     #sets the maximum number of tokens per text
        padding=True,                         #pads shorter sequences to match the longest sequence in the batch
        return_tensors="pt"                     #returns tokenized data as pytorch tensors
    )

    x=x.to("cuda")       #moves tokenized tensors to the gpu
    logits=model(**x).logits      #gets raw prediction scores from the model
    logits=logits.float()          #converts logits to 32-bit floating-point values

    probabilities=torch.softmax(logits,dim=1)          #converts logits into class probabilities
    spoiler_scores=probabilities[:,1].cpu().numpy()       #extracts class 1 probabilities and converts them to a numpy array
    out[j]=spoiler_scores                                 #stores scores at their original text indices

  return out


for n,f in [("val","val.csv"),("test",test_file),("train","train.csv")]:     #iterates through validation, test, and training datasets
 out=D+f"{n}_scored.csv"       #creates the output file path
 if os.path.exists(out):
   print(f"{n} already scored")
   continue                                                            #skips to the next dataset

 path=f if os.path.isabs(f) else os.path.join(D,f)  #uses the full path if f is absolute

 if not os.path.exists(path):
   print(f"{n} file not found: {path}")  #prints the missing file path
   print("Available files:",os.listdir(D))  #lists files in the dataset directory
   continue                       #skips to the next dataset

 d=pd.read_csv(path)  #reads the dataset from the correct path
 d["t1"]=score_with(teacher,ttok,d.text.tolist())    #calculates teacher spoiler probabilities for each text
 d.to_csv(out,index=False)
 print(f"{n} scored {len(d)}")

del teacher        #deletes the teacher model variable
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

val already scored
test already scored
train already scored


**create new clean dataset**


In [45]:
def clean_dataset(df,extra=None):
 spoilers=df[df.label==1]
 non_spoilers=df[df.label==0]

 spoiler_cutoff=keep_pos #selecting those rows only where teacher thinks has the highest spoiler
 spoilers=spoilers[spoilers.t1>=spoiler_cutoff]
 non_spoiler_cutoff=non_spoilers.t1.quantile(keep_neg)
 non_spoilers=non_spoilers[non_spoilers.t1<=non_spoiler_cutoff]   #rows where the teacher thinks has the lowest spoielrs

 if extra is not None:   #extra rows from the movie file (Cell 5c)
  spoilers=pd.concat([spoilers,extra[extra.label==1]])
  non_spoilers=pd.concat([non_spoilers,extra[extra.label==0]])

 n=min(len(spoilers),len(non_spoilers))     #equal rows in both classes
 spoilers=spoilers.sample(n,random_state=seed)
 non_spoilers=non_spoilers.sample(n,random_state=seed)


 both=pd.concat([spoilers,non_spoilers])
 both=both.sample(frac=1,random_state=seed)    #shuffles them

 return both[["text","label","t1"]].reset_index(drop=True)


clean={}
for name in ["train","val","test"]:
 scored=pd.read_csv(D+name+"_scored.csv")
 clean[name]=clean_dataset(scored)
 clean[name].to_csv(D+"clean_"+name+".csv",index=False)
 print(f"{name} rows {len(clean[name])} | per class: {len(clean[name])//2}")

train rows 6140 | per class: 3070
val rows 648 | per class: 324
test rows 770 | per class: 385


In [46]:
train_rows = clean["train"]     #checking the top 12 spoilers and non spoiler found

for label in [1, 0]:
    print("\nlabel", label, "(1 = spoiler, 0 = safe\n")
    sample = train_rows[train_rows.label == label].sample(12, random_state=3)
    for text, score in zip(sample.text, sample.t1):
        print(round(score, 3), "|", text[:170])


label 1 (1 = spoiler, 0 = safe

0.699 | Carlos Verone, Monica Vuentas and Mia Toretto. I personally thought the twist at the end with Dom was a perfect ending despite not actually seeing the race. I have also f
0.729 | Not only ARE these people (partially) responsible for Lucie's degradation, but moments later Lucie is killed, and it is Anna that becomes the centre of attention in a nar
0.668 | I love how we are lead on to believe that someone is A to them have them killed off or lots of evidence at the end showing that they can't be A which then makes you want 
0.549 | Billy ripped her eye out,so why is she doing that?
0.506 | There's even the inevitable rush at the end as the person whose made a series of relationship mistakes runs to what she believes to be the true love of her life and vice 
0.538 | After heroic sacrifices and a major blow to the ship's environment, we're told that half the crew will have to die for there to be enough oxygen to reach the other vessel
0.513 | There

In [50]:
import json
import hashlib
import re
from transformers import AutoTokenizer,AutoModelForSequenceClassification

teacher_id="bhavyagiri/roberta-base-finetuned-imdb-spoilers"     #model name only, "teacher" is the model variable in cell 4
anchor_max=3000     #extra rows per class, at most
anchor_pool=8000     #candidates per class, the teacher keeps the ones it agrees with


def split_of(movie_id):
 h=int(hashlib.md5(movie_id.encode()).hexdigest(),16)%100     #same split as the dataset so val and test movies stay untouched
 if h<80:
  return "train"
 if h<90:
  return "val"
 return "test"


def split_sentences(text):
 text=re.sub(r"\s+"," ",text or "")
 text=re.sub(r"([a-z0-9\)\"'])([.!?])(?=[A-Z])",r"\1\2 ",text)     #glued sentences like end.Next
 return [s for s in re.split(r"(?<=[.!?])\s+",text) if s]


def is_junk(s):
 s=s.lower()
 for w in ["written by","@","<",">","http","imdb"]:     #credit lines at the end of synopses
  if w in s:
   return True
 return False



spoiler_rows=[]
safe_rows=[]
for line in open(D+"IMDB_movie_details.json",encoding="utf-8"):
 movie=json.loads(line)
 if split_of(movie["movie_id"])!="train":
  continue



 synopsis=split_sentences(movie.get("plot_synopsis"))
 for i,s in enumerate(synopsis):
  if not 40<=len(s)<=500 or is_junk(s):
   continue
  if i>=0.7*len(synopsis):
   spoiler_rows.append((s,1))     #last 30% of the synopsis, what happens and how it ends
  elif i<0.25*len(synopsis):
   safe_rows.append((s,0))     #first 25%, setup



 for s in split_sentences(movie.get("plot_summary")):
  if 40<=len(s)<=500 and not is_junk(s):
   safe_rows.append((s,0))     #short summary, premise only


print("found",len(spoiler_rows),"spoiler rows and",len(safe_rows),"safe rows")



pool=pd.concat([
 pd.DataFrame(spoiler_rows,columns=["text","label"]).sample(anchor_pool,random_state=seed),
 pd.DataFrame(safe_rows,columns=["text","label"]).sample(anchor_pool,random_state=seed)]).reset_index(drop=True)



teacher_tok=AutoTokenizer.from_pretrained(teacher_id)
teacher_model=AutoModelForSequenceClassification.from_pretrained(teacher_id,use_safetensors=True).half().cuda().eval()


texts=pool.text.tolist()
scores=[]
for start in range(0,len(texts),128):
 enc=teacher_tok(texts[start:start+128],truncation=True,max_length=maxlen,padding=True,return_tensors="pt").to("cuda")
 with torch.no_grad():
  logits=teacher_model(**enc).logits
 scores+=torch.softmax(logits.float(),dim=1)[:,1].cpu().tolist()
pool["t1"]=scores



del teacher_model
torch.cuda.empty_cache()

#same rule as the review rows, keep only what the teacher agrees with
spoiler_anchors=pool[(pool.label==1)&(pool.t1>=keep_pos)]
safe_anchors=pool[(pool.label==0)&(pool.t1<=pool[pool.label==0].t1.quantile(keep_neg))]
print("teacher kept",len(spoiler_anchors),"spoiler rows and",len(safe_anchors),"safe rows")

spoiler_anchors=spoiler_anchors.sample(min(anchor_max,len(spoiler_anchors)),random_state=seed)
safe_anchors=safe_anchors.sample(min(anchor_max,len(safe_anchors)),random_state=seed)
anchors=pd.concat([spoiler_anchors,safe_anchors])

train_scored=pd.read_csv(D+"train_scored.csv")
clean["train"]=clean_dataset(train_scored,extra=anchors)
clean["train"].to_csv(D+"clean_train.csv",index=False)
print(f"train rows {len(clean['train'])} | per class: {len(clean['train'])//2}")

found 26621 spoiler rows and 28796 safe rows


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

teacher kept 1942 spoiler rows and 7200 safe rows
train rows 10024 | per class: 5012


In [51]:
import os

print([f for f in os.listdir(D) if f.endswith(".json")])

['IMDB_reviews.json', 'IMDB_movie_details.json']
